# Entrenamiento Transformer

En este notebook se entrena el transformer diseñado en transformer.py

In [ ]:
#Importacion librerias
from google.colab import drive
from google.colab import files
from tqdm.notebook import tqdm
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torchvision.transforms import ToTensor
import sys
from sklearn.model_selection import train_test_split

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es')


Mounted at /content/drive


In [ ]:
#Rutas donde se encuentran nuestros propios .py
from dataset import tiktoken
from dataset import Dataset
from dataset import Tokenizer
from Transformer import Transformer

dataset_path = "/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es/dataset_800k.json"
pesos_patht ="/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es/resultados/"
vocab_path = "/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es/vocabulario_32k.json"
results_path = "/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es/resultados/"



Cargamos el dataset en un dataframe y dividimos 80% entrenamiento y 20% validacion

In [ ]:
# Crear dataset y dataloader
df = pd.read_json(dataset_path)

train, val = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

print(len(train))
print(len(val))

720000
80000


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Mar 15 19:09:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   43C    P0             54W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
tokenizer = Tokenizer(vocab_path)

train_loader  =  DataLoader(Dataset(train,tokenizer), batch_size=1040, shuffle=True)
val_loader = DataLoader(Dataset(val,tokenizer), batch_size=8, shuffle=False)
print(len(tokenizer))
transformer = Transformer(vocab_size=len(tokenizer),pad_id= tokenizer.pad_id()).to(device=device)


32000


In [ ]:
#Cargar pesos para continuar entrenamiento
weights_path = results_path + "1_val_1.0604.pth"
weights  = torch.load(weights_path, map_location=device, weights_only=True)
transformer.load_state_dict(weights,strict=False)

_IncompatibleKeys(missing_keys=[], unexpected_keys=['encoder_blocks.0.attention_heads.block_attention.0.tril', 'encoder_blocks.0.attention_heads.block_attention.1.tril', 'encoder_blocks.0.attention_heads.block_attention.2.tril', 'encoder_blocks.0.attention_heads.block_attention.3.tril', 'encoder_blocks.0.attention_heads.block_attention.4.tril', 'encoder_blocks.0.attention_heads.block_attention.5.tril', 'encoder_blocks.0.attention_heads.block_attention.6.tril', 'encoder_blocks.0.attention_heads.block_attention.7.tril', 'encoder_blocks.1.attention_heads.block_attention.0.tril', 'encoder_blocks.1.attention_heads.block_attention.1.tril', 'encoder_blocks.1.attention_heads.block_attention.2.tril', 'encoder_blocks.1.attention_heads.block_attention.3.tril', 'encoder_blocks.1.attention_heads.block_attention.4.tril', 'encoder_blocks.1.attention_heads.block_attention.5.tril', 'encoder_blocks.1.attention_heads.block_attention.6.tril', 'encoder_blocks.1.attention_heads.block_attention.7.tril', 'enc

ENTRENAMIENTO


In [ ]:
import torch
from torch.nn import functional as F
import math
from torch.optim.lr_scheduler import LambdaLR
import numpy as np

  #Par de frases para ver comportamiento
frases_test = [
"I'm so happy to see you!",
"Where are you going right now?",
"Stop talking and listen to me!"]


def check_behavior():
  #Prueba comportamiento
  for i, frase in enumerate(frases_test):

    x_tokens = torch.tensor(np.array([tokenizer.encode(frase, pad=False)])).to('cuda')
    y_tokens = torch.tensor(np.array([[tokenizer.star_of_text_id()]])).to('cuda')

    output = transformer.predict(x_tokens, y_tokens,end_token_id=tokenizer.end_of_text_id(),device='cuda')

    #output[0] porque es un batch de 1)
    traduccion = tokenizer.decoder(output[0])
    print(f"| {frase:<45} | {traduccion:<45}")


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, num_cycles=0.5):
    """
    Crea un scheduler que sube el LR linealmente desde 0 hasta el LR máximo durante el warmup,
    y luego lo reduce siguiendo la mitad de una onda coseno (baja hasta 0 al final).
    """
    def lr_lambda(current_step):
        # Fase 1: Warmup (subida lineal)
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))

        # Fase 2: Decaimiento Coseno (bajada suave)
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress)))

    return LambdaLR(optimizer, lr_lambda)

@torch.no_grad()
def estimate_loss(eval_iters=5):
    out = {}
    transformer.eval() # Ponemos el modelo en modo evaluación (desactiva Dropout, etc.)

    loaders = {
        'train': train_loader,
        'val': val_loader
    }

    for split, loader in loaders.items():
        losses = torch.zeros(eval_iters)
        data_iter = iter(loader)

        for k in range(eval_iters):
            try:
                x_text, y_full_text = next(data_iter)
            except StopIteration:
                data_iter = iter(loader)
                x_text, y_full_text = next(data_iter)

            # 1. Tokenizamos igual que en el entrenamiento
            x_tokens = tokenizer.encode_batch(x_text).to(device)
            y_tokens = tokenizer.encode_batch(y_full_text).to(device)

            # 2. Desplazamiento (Teacher Forcing)
            y_input = y_tokens[:, :-1]
            y_target = y_tokens[:, 1:]

            # 3. Forward pass
            logits = transformer(x_tokens, y_input)

            # 4. Ajuste de dimensiones
            B, T, C = logits.shape
            logits = logits.reshape(B * T, C)
            y_target = y_target.reshape(B * T)


            # 5. Cálculo de la pérdida IGNORANDO el <PAD>
            loss = F.cross_entropy(logits, y_target, ignore_index=tokenizer.pad_id())
            losses[k] = loss.item()


        out[split] = losses.mean().item()



    check_behavior()

    transformer.train() # Volvemos a modo entrenamiento
    return out

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm


LEARNING_RATE = 5e-4
EVAL_ITERS = 1
epochs = 20

model = transformer.to(device)
model.train()

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id())
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


# Calculamos el total de pasos (batches por época * número de épocas)
total_steps = len(train_loader) * epochs

# el 10% del entrenamiento para warmup
warmup_steps = int(total_steps * 0.1)

scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)


for epoch in tqdm(range(epochs), desc="Training"):
    total_loss = 0
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{epochs}", leave=True)

    if (epoch) % EVAL_ITERS == 0 or epoch == epochs - 1:
        losses = estimate_loss()
        torch.save(model.state_dict(), pesos_patht + f"{epoch}_val_{losses['val']:.4f}.pth")
        tqdm.write(f"Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

    for batch_idx, (x, y_full) in progress_bar:


      x = tokenizer.encode_batch(x).to(device)
      y_full = tokenizer.encode_batch(y_full).to(device)

      y_input = y_full[:, :-1]
      y_target = y_full[:, 1:]

      logits = model(x, y_input)

      B, T, C = logits.shape
      logits = logits.reshape(B*T, C)
      y_target = y_target.reshape(B*T)

      loss = criterion(logits, y_target)
      total_loss += loss.item()

      optimizer.zero_grad(set_to_none=True)
      loss.backward()

      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

      optimizer.step()

        # Actualiza learning rate
      scheduler.step()


      current_lr = scheduler.get_last_lr()[0]
      progress_bar.set_postfix({
          'loss': f'{loss.item():.4f}',
          'lr': f'{current_lr:.6f}' # Lo mostramos para que veas cómo sube y baja
      })

    avg_loss = total_loss / len(train_loader)
    tqdm.write(f"Loss Promedio: {avg_loss:.4f}\n")

Training:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1/20:   0%|          | 0/693 [00:00<?, ?it/s]

| I'm so happy to see you!                      | ¡Me da mucho gusto verte!                    
| Where are you going right now?                | ¿Dónde vas ahora mismo?                      
| Stop talking and listen to me!                | ¡Deja de hablar y escme!                     
Train Loss: 1.1514 | Val Loss: 1.4267
Loss Promedio: 0.9582



Epoch 2/20:   0%|          | 0/693 [00:00<?, ?it/s]

| I'm so happy to see you!                      | Estoy tan feliz de verte!                    
| Where are you going right now?                | ¿Adónde vas ahora?                           
| Stop talking and listen to me!                | Deja de hablar y escagueme.                  
Train Loss: 0.8864 | Val Loss: 1.2752
Loss Promedio: 1.0053



Epoch 3/20:   0%|          | 0/693 [00:00<?, ?it/s]

| I'm so happy to see you!                      | ¡Estoy tanto feliz de verte!                 
| Where are you going right now?                | ¿Dónde está pasando ahora?                   
| Stop talking and listen to me!                | ¡Deja de hablarme.                           
Train Loss: 0.9502 | Val Loss: 1.3587


KeyboardInterrupt: 